# Physics Simulation Starter

This notebook sets up a simple Python environment for running basic physics simulations.

## What this notebook actually does, in plain terms

This notebook builds up, piece by piece, a simulated version of the plasma pulse propulsion idea: hold a "packet" of charged plasma in a ring-shaped magnetic trap, then open a gap in that trap at exactly the right moment so the packet shoots out as a directed pulse.

Roughly, it goes through:
1. **Shape** -- build the physical layout (a torus/ring), no physics yet
2. **A first fake magnetic field** -- a rough sketch, just to check the visuals work
3. **A real magnetic field** -- calculate what field the actual coils would produce, using the law behind every electromagnet (Biot-Savart)
4. **A gate** -- a second, switchable coil that punches a hole in the field on one side
5. **A test particle** -- fire a single charged particle through the field and watch where it goes
6. **Does it actually stay trapped?** -- the newest section, checking whether the ring can hold a particle at all, before worrying about the gate

No equation below is assumed knowledge going in -- each one gets a plain-language version first.

### Building the shape: two nested rings

Picture two donuts, one inside the other, sharing the same centre line:
- the **inner ring (cyan)** is the plasma channel -- the space where the charged plasma packet actually lives and travels
- the **outer ring (orange)** is the coil former -- the shape the electromagnet wire is wound around

`R` is the size of the whole ring (radius from the centre of the donut hole out to the middle of the tube). `r_plasma` and `r_coil` are how fat each tube is. Nothing physical happens yet -- this just draws the geometry so everything after has a shape to sit inside.

In [33]:
import numpy as np
import pyvista as pv

pv.set_jupyter_backend("trame")  # keep your working backend

# --- Torus parameters ---
R = 0.125   # major radius
r_plasma = 0.03   # plasma minor radius
r_coil   = 0.07   # coil pack minor radius

# --- Create torus meshes ---
plasma_torus = pv.ParametricTorus(R, r_plasma)
coil_torus   = pv.ParametricTorus(R, r_coil)

# --- Plot both ---
plotter = pv.Plotter()
plotter.add_mesh(plasma_torus, color="cyan", opacity=0.6)
plotter.add_mesh(coil_torus, color="orange", opacity=0.3)
plotter.add_axes()
plotter.show()



Widget(value='<iframe src="http://localhost:60710/index.html?ui=P_0x2b5bba25810_26&reconnect=auto" class="pyvi…

### How much "push" are the coils giving?

An electromagnet's strength depends on two things multiplied together: how many times the wire is wound around (**turns**, `N`) and how much current flows through each wind (`I`, in amps). Multiply them and you get **ampere-turns** (`NI`) -- the standard way to compare electromagnet strength, whether it's a few turns at high current or many turns at low current.

`J` spreads that same ampere-turn total over the coil's cross-sectional area -- current density, i.e. how "packed" the current is. Not used downstream yet; this cell just establishes the numbers.

In [34]:
import numpy as np

# --- Coil parameters ---
N = 200          # turns
I = 2200         # current per turn (A)
NI = N * I       # total ampere-turns

# --- Coil cross-section area ---
r_plasma = 0.03
r_coil   = 0.07

A_coil = np.pi * (r_coil**2 - r_plasma**2)

# --- Current density (A/m^2) ---
J = NI / A_coil

J


35014087.480216965

### First guess at a field -- and it's deliberately wrong

Before any real physics, this cell paints a single arrow direction (+Y) at every point on the plasma ring, just to check the 3D field-arrow plotting works. It is **not** what a real field around a ring looks like -- a uniform field pointing one flat direction doesn't curl around a torus at all. Think of it as testing the paintbrush before painting the actual picture.

In [35]:
import numpy as np
import pyvista as pv

pv.set_jupyter_backend("trame")

# --- Geometry (reuse from before) ---
R = 0.125
r_plasma = 0.03
r_coil   = 0.07

plasma_torus = pv.ParametricTorus(R, r_plasma)
coil_torus   = pv.ParametricTorus(R, r_coil)

# --- Sample points on the plasma torus surface ---
points = plasma_torus.points  # (N, 3) array of XYZ

# --- Dummy toroidal B-field: constant magnitude, tangent to torus ---
# For now, we just set a simple vector field that points along +Y everywhere
B = np.tile(np.array([0.0, 0.7, 0.0]), (points.shape[0], 1))  # 0.7 T along +Y

# --- Create a PyVista point cloud with vectors ---
cloud = pv.PolyData(points)
cloud["B"] = B

# --- Plot geometry + field vectors ---
plotter = pv.Plotter()
plotter.add_mesh(plasma_torus, color="cyan", opacity=0.6)
plotter.add_mesh(coil_torus, color="orange", opacity=0.3)
plotter.add_mesh(cloud.glyph(orient="B", scale="B", factor=0.08), color="red")
plotter.add_axes()
plotter.show()


Widget(value='<iframe src="http://localhost:60710/index.html?ui=P_0x2b5bba25e50_27&reconnect=auto" class="pyvi…

### Making the arrows actually wrap around the ring

A field meant to hold something in a ring shape needs to point **along** the ring at every point, not in one fixed direction. This cell fixes that: at each point it works out the angle around the ring (`phi`) and points the field tangent to the circle at that angle, so the arrows curl continuously all the way round -- the way a field wrapping a torus actually should. This is the **toroidal direction**: "toroidal" means *the long way around the donut*, as opposed to "poloidal" -- *the short way around, through the hole* -- which matters later.

Still a hand-picked number (`B0 = 0.7` T) here, not yet calculated from the coils themselves.

In [36]:
import numpy as np
import pyvista as pv

pv.set_jupyter_backend("trame")

# --- Geometry ---
R = 0.125
r_plasma = 0.03
r_coil   = 0.07

plasma_torus = pv.ParametricTorus(R, r_plasma)
coil_torus   = pv.ParametricTorus(R, r_coil)

# --- Sample points on the plasma torus ---
points = plasma_torus.points  # (N, 3)

# --- Compute toroidal field direction at each point ---
B0 = 0.7  # Tesla

B = np.zeros_like(points)

for i, (x, y, z) in enumerate(points):
    phi = np.arctan2(y, x)
    # unit vector in toroidal direction
    B[i] = B0 * np.array([-np.sin(phi), np.cos(phi), 0.0])

# --- Attach field to point cloud ---
cloud = pv.PolyData(points)
cloud["B"] = B

# --- Plot ---
plotter = pv.Plotter()
plotter.add_mesh(plasma_torus, color="cyan", opacity=0.6)
plotter.add_mesh(coil_torus, color="orange", opacity=0.3)
plotter.add_mesh(cloud.glyph(orient="B", scale="B", factor=0.08), color="red")
plotter.add_axes()
plotter.show()


Widget(value='<iframe src="http://localhost:60710/index.html?ui=P_0x2b5bba25950_28&reconnect=auto" class="pyvi…

### Giving the field an edge

Real magnetic fields aren't uniform everywhere -- they're strong where the coils are and fall off elsewhere. This cell is a rough stand-in for that: it checks whether a point is inside the plasma channel or not, and only "turns on" the toroidal field if it is. Still a placeholder (the strength is still the same hand-picked `0.7` T, not calculated from actual coil current) -- but it's the first cell with a real inside/outside boundary, which matters later.

In [37]:
import numpy as np
import pyvista as pv

pv.set_jupyter_backend("trame")

# --- Geometry parameters ---
R = 0.125
r_plasma = 0.03
r_coil   = 0.07

# --- Coil parameters ---
N = 200
I = 2200
NI = N * I
A_coil = np.pi * (r_coil**2 - r_plasma**2)
J = NI / A_coil   # current density

# --- Create torus meshes ---
plasma_torus = pv.ParametricTorus(R, r_plasma)
coil_torus   = pv.ParametricTorus(R, r_coil)

# --- Field solver scaffold ---
def B_field(x, y, z):
    """
    Placeholder magnetic field solver.
    Returns a toroidal field inside the plasma region,
    and zero outside.
    """
    # distance from torus centerline
    r_xy = np.sqrt(x**2 + y**2)
    dist_from_major = np.sqrt((r_xy - R)**2 + z**2)

    # inside plasma region
    if dist_from_major < r_plasma:
        phi = np.arctan2(y, x)
        B0 = 0.7  # Tesla
        return np.array([-np.sin(phi)*B0, np.cos(phi)*B0, 0.0])

    # outside plasma region
    return np.array([0.0, 0.0, 0.0])

# --- Sample grid around torus ---
grid_x = np.linspace(-0.35, 0.35, 40)
grid_y = np.linspace(-0.35, 0.35, 40)
grid_z = np.linspace(-0.20, 0.20, 20)

points = np.array([[x, y, z] for x in grid_x for y in grid_y for z in grid_z])

# --- Compute B-field on grid ---
B = np.array([B_field(x, y, z) for x, y, z in points])

# --- Visualise ---
cloud = pv.PolyData(points)
cloud["B"] = B

plotter = pv.Plotter()
plotter.add_mesh(plasma_torus, color="cyan", opacity=0.6)
plotter.add_mesh(coil_torus, color="orange", opacity=0.3)
plotter.add_mesh(cloud.glyph(orient="B", scale="B", factor=0.08), color="red")
plotter.add_axes()
plotter.show()


Widget(value='<iframe src="http://localhost:60710/index.html?ui=P_0x2b5bba25a90_29&reconnect=auto" class="pyvi…

### From guesswork to the real law: Biot-Savart

This is the first cell that calculates an actual, physically correct field rather than assuming one. The law behind it -- **Biot-Savart** -- is the same rule behind every electromagnet, transformer, and MRI machine: *every tiny piece of current-carrying wire creates a small magnetic field, and you add up the contributions from every piece of wire to get the total field at any point.*

Concretely, the code:
- builds 20 real wire loops arranged around the ring (`segments`), each shaped like a small circle -- this is what a real toroidal-field coil pack looks like
- chops each loop into tiny straight segments (`dl`)
- for each point in space, sums the tiny field contribution from every segment of every loop

The formula in the code (`mu0 * I / (4*pi) * cross(dl, r_vec) / dist**3`) is just that idea written as maths: more current or more nearby wire -> bigger contribution; further away -> weaker (the `dist**3` is why field strength drops off quickly with distance). `mu0` is a fixed constant of nature (how easily magnetic fields form in empty space) that shows up in every magnetism equation -- not something you tune.

In [38]:
import numpy as np
import pyvista as pv

pv.set_jupyter_backend("trame")

mu0 = 4e-7 * np.pi

# --- Geometry ---
R = 0.125
r_plasma = 0.03
r_coil   = 0.07

# --- Coil parameters ---
N_loops = 20        # reduced for speed
I = 2200
theta_samples = 200

# --- Precompute loops, dl, and midpoints ---
phi_vals = np.linspace(0, 2*np.pi, N_loops, endpoint=False)
theta_vals = np.linspace(0, 2*np.pi, theta_samples)

segments = []

for phi in phi_vals:
    loop = []
    for theta in theta_vals:
        x = (R + r_coil * np.cos(theta)) * np.cos(phi)
        y = (R + r_coil * np.cos(theta)) * np.sin(phi)
        z = r_coil * np.sin(theta)
        loop.append([x, y, z])
    loop = np.array(loop)

    # segment endpoints
    p1 = loop
    p2 = np.roll(loop, -1, axis=0)

    dl = p2 - p1
    mid = 0.5 * (p1 + p2)

    segments.append((p1, dl, mid))

# --- Vectorised Biot–Savart ---
def B_field_fast(x, y, z):
    r = np.array([x, y, z])
    B = np.zeros(3)

    for p1, dl, mid in segments:
        r_vec = r - mid
        dist = np.linalg.norm(r_vec, axis=1)
        mask = dist > 1e-6

        dB = mu0 * I / (4*np.pi) * np.cross(dl[mask], r_vec[mask]) / (dist[mask]**3)[:, None]
        B += dB.sum(axis=0)

    return B

# --- Sample grid ---
grid_x = np.linspace(-0.35, 0.35, 40)
grid_y = np.linspace(-0.35, 0.35, 40)
grid_z = np.linspace(-0.20, 0.20, 20)

points = np.array([[x, y, z] for x in grid_x for y in grid_y for z in grid_z])

# --- Compute B-field ---
B = np.array([B_field_fast(x, y, z) for x, y, z in points])

# --- Visualise ---
cloud = pv.PolyData(points)
cloud["B"] = B

plasma_torus = pv.ParametricTorus(R, r_plasma)
coil_torus   = pv.ParametricTorus(R, r_coil)

plotter = pv.Plotter()
plotter.add_mesh(plasma_torus, color="cyan", opacity=0.6)
plotter.add_mesh(coil_torus, color="orange", opacity=0.3)
plotter.add_mesh(cloud.glyph(orient="B", scale="B", factor=0.08), color="red")
plotter.add_axes()
plotter.show()


Widget(value='<iframe src="http://localhost:60710/index.html?ui=P_0x2b53e91fed0_30&reconnect=auto" class="pyvi…

### Adding a second, smaller magnet: the nozzle

Everything so far builds one continuous ring of field. To let the packet out on demand, you need a second, separate coil positioned at just one spot on the ring (`nozzle_center`) -- a smaller electromagnet placed like a gate, using the same Biot-Savart approach as the main coils. On its own here it's always on, just added straight onto the main field (`B_total`); the next cell is what turns it into an actual switch.

In [41]:
import numpy as np

# --- Nozzle coil parameters ---
nozzle_radius = 0.05
nozzle_current = 20000
nozzle_theta = np.linspace(0, 2*np.pi, 200)

# Position nozzle coil at φ = 0 (front of torus)
nozzle_center = np.array([R + r_coil + 0.01, 0.0, 0.0])  # slight outward offset

# Compute nozzle coil loop points
nozzle_loop = np.array([
    nozzle_center + np.array([
        nozzle_radius * np.cos(t),
        0.0,
        nozzle_radius * np.sin(t)
    ])
    for t in nozzle_theta
])

# Precompute nozzle segments
nozzle_p1 = nozzle_loop
nozzle_p2 = np.roll(nozzle_loop, -1, axis=0)
nozzle_dl = nozzle_p2 - nozzle_p1
nozzle_mid = 0.5 * (nozzle_p1 + nozzle_p2)

def B_field_nozzle(x, y, z):
    r = np.array([x, y, z])
    B = np.zeros(3)

    r_vec = r - nozzle_mid
    dist = np.linalg.norm(r_vec, axis=1)
    mask = dist > 1e-6

    dB = mu0 * nozzle_current / (4*np.pi) * np.cross(nozzle_dl[mask], r_vec[mask]) / (dist[mask]**3)[:, None]
    B += dB.sum(axis=0)

    return B

# --- Combined field ---
def B_total(x, y, z):
    return B_field_fast(x, y, z) + B_field_nozzle(x, y, z)

# --- Compute combined field on grid ---
B_combined = np.array([B_total(x, y, z) for x, y, z in points])

# --- Visualise ---
cloud = pv.PolyData(points)
cloud["B"] = B_combined

plotter = pv.Plotter()
plotter.add_mesh(plasma_torus, color="cyan", opacity=0.6)
plotter.add_mesh(coil_torus, color="orange", opacity=0.3)
plotter.add_mesh(cloud.glyph(orient="B", scale="B", factor=0.08), color="red")

# Visualise nozzle coil
plotter.add_mesh(pv.PolyData(nozzle_loop), color="green", point_size=5)

plotter.add_axes()
plotter.show()


Widget(value='<iframe src="http://localhost:60710/index.html?ui=P_0x2b5bba25f90_33&reconnect=auto" class="pyvi…

### Turning the nozzle into a gate that opens and closes

`nozzle_current_t(t)` is a simple on/off switch: current flows for the first half of each cycle, then cuts to zero for the second half (a **square wave**), at a rate of `f_switch` cycles per second. Everywhere else the code is identical to the last cell, just recomputed at several snapshots in time (`t_vals`) so you can watch the nozzle's local field appear and disappear across frames while the main ring field stays constant.

In [42]:
import numpy as np
import pyvista as pv

pv.set_jupyter_backend("trame")

# --- Time settings ---
t_vals = np.linspace(0.0, 1.0, 6)   # 6 frames from t=0 to t=1
f_switch = 2.0                      # 2 Hz switching frequency

# --- Switching waveform: square wave ---
def nozzle_current_t(t):
    phase = (t * f_switch) % 1.0
    return nozzle_current if phase < 0.5 else 0.0


# --- Time-dependent nozzle field ---
def B_field_nozzle_t(x, y, z, t):
    r = np.array([x, y, z])
    B = np.zeros(3)

    I_t = nozzle_current_t(t)
    if I_t == 0.0:
        return B

    r_vec = r - nozzle_mid
    dist = np.linalg.norm(r_vec, axis=1)
    mask = dist > 1e-6

    dB = mu0 * I_t / (4*np.pi) * np.cross(nozzle_dl[mask], r_vec[mask]) / (dist[mask]**3)[:, None]
    B += dB.sum(axis=0)

    return B


# --- Combined field (toroidal + nozzle) ---
def B_total_t(x, y, z, t):
    return B_field_fast(x, y, z) + B_field_nozzle_t(x, y, z, t)


# --- Loop over time frames and plot ---
for t in t_vals:
    print(f"Rendering frame at t = {t:.2f} s")

    B_frame = np.array([B_total_t(x, y, z, t) for x, y, z in points])

    cloud = pv.PolyData(points)
    cloud["B"] = B_frame

    plotter = pv.Plotter()
    plotter.add_mesh(plasma_torus, color="cyan", opacity=0.6)
    plotter.add_mesh(coil_torus, color="orange", opacity=0.3)
    plotter.add_mesh(cloud.glyph(orient="B", scale="B", factor=0.08), color="red")

    # nozzle coil
    plotter.add_mesh(pv.PolyData(nozzle_loop), color="green", point_size=5)

    plotter.add_axes()
    plotter.show(title=f"t = {t:.2f} s")



Rendering frame at t = 0.00 s


Widget(value='<iframe src="http://localhost:60710/index.html?ui=P_0x2b5dc886d50_34&reconnect=auto" class="pyvi…

Rendering frame at t = 0.20 s


Widget(value='<iframe src="http://localhost:60710/index.html?ui=P_0x2b5dc885950_35&reconnect=auto" class="pyvi…

Rendering frame at t = 0.40 s


...

Widget(value='<iframe src="http://localhost:60710/index.html?ui=P_0x2b604a18190_37&reconnect=auto" class="pyvi…

Rendering frame at t = 0.80 s


Widget(value='<iframe src="http://localhost:60710/index.html?ui=P_0x2b604a18a50_38&reconnect=auto" class="pyvi…

Rendering frame at t = 1.00 s


Widget(value='<iframe src="http://localhost:60710/index.html?ui=P_0x2b604a19310_39&reconnect=auto" class="pyvi…

### Chasing a single charged particle: Lorentz force and RK4

This is where a plasma packet is actually simulated moving through the field, rather than just drawing arrows.

**The physics:** a moving charged particle in a magnetic field feels a sideways push -- the **Lorentz force** -- always perpendicular to both its velocity and the field direction (`q_over_m * cross(v, B)` in the code). This is exactly why charged particles spiral around field lines instead of flying straight through them: the push never speeds them up or slows them down, only ever curves their path.

**Why RK4:** once the field varies with position and time, there's no single clean equation for "where does the particle end up" -- the code steps forward in tiny time increments instead, recalculating the force each step. **RK4** (Runge-Kutta 4th order) is a more accurate way of taking each small step than the simplest method: instead of using the force at just the start of the step, it samples the force at four points across the step (start, two midpoints, end) and blends them, which keeps small errors from snowballing over thousands of steps. You don't need to follow the four `k1`-`k4` terms individually -- just that it's a well-established, reliable way to trace a curving path accurately.

In [ ]:
import numpy as np
import pyvista as pv

pv.set_jupyter_backend("trame")

# --- Plasma packet properties ---
q_over_m = 5e5        # stronger Lorentz response
dt = 1e-5             # time step
n_steps = 3000        # longer run so packet can leave

# --- Initial conditions (shifted toward nozzle side) ---
x0 = R + 0.02         # slightly toward nozzle
y0 = 0.0
z0 = 0.0

# Initial velocity with outward bias
v0 = np.array([300.0, 300.0, 0.0])

r = np.array([x0, y0, z0])
v = v0.copy()

# --- Time-dependent field (higher switching frequency) ---
f_switch = 5.0        # gate opens/closes more often

def nozzle_current_t(t):
    phase = (t * f_switch) % 1.0
    return nozzle_current if phase < 0.5 else 0.0

def B_field_nozzle_t(x, y, z, t):
    r_vec = np.array([x, y, z])
    B = np.zeros(3)

    I_t = nozzle_current_t(t)
    if I_t == 0.0:
        return B

    diff = r_vec - nozzle_mid
    dist = np.linalg.norm(diff, axis=1)
    mask = dist > 1e-6

    dB = mu0 * I_t / (4*np.pi) * np.cross(nozzle_dl[mask], diff[mask]) / (dist[mask]**3)[:, None]
    B += dB.sum(axis=0)
    return B

def B_total_t(x, y, z, t):
    return B_field_fast(x, y, z) + B_field_nozzle_t(x, y, z, t)

# --- Lorentz force integrator (RK4) ---
def B_at(r_vec, t):
    return B_total_t(r_vec[0], r_vec[1], r_vec[2], t)

def accel(r_vec, v_vec, t):
    return q_over_m * np.cross(v_vec, B_at(r_vec, t))

def rk4_step(r_vec, v_vec, t, dt):
    a1 = accel(r_vec, v_vec, t)
    k1_v = a1 * dt
    k1_r = v_vec * dt

    a2 = accel(r_vec + 0.5*k1_r, v_vec + 0.5*k1_v, t + 0.5*dt)
    k2_v = a2 * dt
    k2_r = (v_vec + 0.5*k1_v) * dt

    a3 = accel(r_vec + 0.5*k2_r, v_vec + 0.5*k2_v, t + 0.5*dt)
    k3_v = a3 * dt
    k3_r = (v_vec + 0.5*k2_v) * dt

    a4 = accel(r_vec + k3_r, v_vec + k3_v, t + dt)
    k4_v = a4 * dt
    k4_r = (v_vec + k3_v) * dt

    v_next = v_vec + (k1_v + 2*k2_v + 2*k3_v + k4_v) / 6.0
    r_next = r_vec + (k1_r + 2*k2_r + 2*k3_r + k4_r) / 6.0

    return r_next, v_next

# --- Integrate trajectory ---
t = 0.0
traj = []

for i in range(n_steps):
    traj.append(r.copy())
    r, v = rk4_step(r, v, t, dt)
    t += dt

traj = np.array(traj)

# --- Relaxed clipping so escape is visible ---
traj_clipped = traj[np.linalg.norm(traj, axis=1) < 0.8]

# --- Convert trajectory to sphere glyphs ---
traj_points = pv.PolyData(traj_clipped)
traj_points["ones"] = np.ones(len(traj_clipped))
sphere = pv.Sphere(radius=0.01)

# --- Field snapshot ---
t_mid = 0.5
B_snapshot = np.array([B_total_t(x, y, z, t_mid) for x, y, z in points])
cloud = pv.PolyData(points)
cloud["B"] = B_snapshot

# --- Plot ---
plotter = pv.Plotter()
plotter.add_mesh(plasma_torus, color="cyan", opacity=0.5)
plotter.add_mesh(coil_torus, color="orange", opacity=0.3)
plotter.add_mesh(cloud.glyph(orient="B", scale="B", factor=0.06), color="red")
plotter.add_mesh(pv.PolyData(nozzle_loop), color="green", point_size=5)

# Cyan dots for trajectory
plotter.add_mesh(
    traj_points.glyph(scale=False, geom=sphere),
    color="cyan"
)

plotter.add_axes()
plotter.show_bounds(grid='front', location='outer', all_edges=True)

# Lock camera
plotter.camera_position = [
    (0.4, 0.4, 0.4),
    (0.0, 0.0, 0.0),
    (0.0, 0.0, 1.0)
]

plotter.show(title="Gate-Escape Plasma Packet (Dots)")


## Confinement Survival Test

Everything above tests whether the packet can be *ejected* through the nozzle gate. It doesn't yet test whether the packet can be *confined* in the first place -- the field so far is purely toroidal, and a purely toroidal field is known not to confine charged particles: grad-B and curvature drift push them off-axis regardless of any gate.

This section adds a poloidal field component (modelling an effective net toroidal plasma current, the same mechanism a real tokamak uses) and tests whether a packet can survive many orbits *without* the nozzle gate at all. If this doesn't hold up, nothing downstream (betatron ramp, RF heating, magnetic mirror compression) is worth building on top of it.

### Plain-English translations for this section

- **Drift** -- a slow sideways creep that happens even though nothing is obviously pushing sideways. It's a side-effect of the field being slightly stronger on one side of the ring than the other (the inner edge of a torus is always more tightly curved than the outer edge) -- like a marble slowly working its way to one side of a bowl that's ever so slightly lopsided, rather than sitting still. Left unchecked, this drift alone eventually carries the particle out of the channel, with no gate involved at all.
- **Gyroradius** -- how wide the little spiral the particle traces around a field line actually is. Small (much smaller than the channel) means the particle is tightly tied to the field line -- well-controlled. Large means the field is barely influencing it at all, regardless of how the rest of the field is shaped. This turned out to be the actual first problem, further down.
- **Toroidal vs poloidal** -- toroidal is the long way around the donut (the direction the packet travels to go all the way round the ring); poloidal is the short way around, through the tube's own cross-section (like tracing the circular cross-section of a bicycle inner tube). A field that's only toroidal has no "twist" to it; adding a poloidal component makes field lines spiral like a candy cane, which is what actually cancels the sideways drift over many laps.

In [ ]:
import numpy as np

# --- Poloidal field from an effective net toroidal plasma current ---
# Real tokamaks get their poloidal field mainly from the plasma's own
# toroidal current (via transformer/solenoid action), not from external
# coils. We model that here as a net current I_p flowing along the torus
# centerline. By Ampere's law this generates a field that circles the
# local minor cross-section:
#
#   inside the channel (uniform current density):
#     B_pol(r_minor) = mu0 * I_p * r_minor / (2*pi*r_plasma**2)
#   outside the channel:
#     B_pol(r_minor) = mu0 * I_p / (2*pi*r_minor)
#
# Combined with the existing toroidal (TF coil) field, this twists field
# lines into helices -- the standard tokamak fix for grad-B/curvature drift.

I_p = 5e4  # effective plasma current, A -- tuned below, see note at the end

def B_poloidal(x, y, z):
    pos = np.array([x, y, z])
    phi = np.arctan2(y, x)

    # nearest point on the torus centerline (magnetic axis)
    centerline = np.array([R * np.cos(phi), R * np.sin(phi), 0.0])

    d = pos - centerline
    r_minor = np.linalg.norm(d)
    if r_minor < 1e-9:
        return np.zeros(3)

    d_hat = d / r_minor
    phi_hat = np.array([-np.sin(phi), np.cos(phi), 0.0])  # local toroidal direction

    pol_hat = np.cross(phi_hat, d_hat)  # circles the centerline
    norm = np.linalg.norm(pol_hat)
    if norm < 1e-12:
        return np.zeros(3)
    pol_hat /= norm

    if r_minor <= r_plasma:
        mag = mu0 * I_p * r_minor / (2*np.pi*r_plasma**2)
    else:
        mag = mu0 * I_p / (2*np.pi*r_minor)

    return mag * pol_hat

def B_confine(x, y, z):
    """Toroidal (TF coil, Biot-Savart) field + poloidal field -> helical field lines."""
    return B_field_fast(x, y, z) + B_poloidal(x, y, z)

### Testing survival: send it around the track and see where it ends up

This cell doesn't add new physics -- it sets up a way to launch a packet on laps of the ring and record, at every step, how far it has drifted from the exact centre-line of the tube (`r_minor`). Think of timing a car around a track and recording how far it wanders from the racing line each lap, rather than whether it crashes outright.

In [ ]:
# --- Confinement survival test ---
# No nozzle/gate field here on purpose -- this isolates whether
# toroidal+poloidal confinement alone can hold a packet across many orbits.

def r_minor_at(pos):
    phi = np.arctan2(pos[1], pos[0])
    centerline = np.array([R*np.cos(phi), R*np.sin(phi), 0.0])
    return np.linalg.norm(pos - centerline)

def run_orbit(B_func, q_over_m, speed, n_orbits, steps_per_orbit, r_offset=0.01, perp_frac=0.15):
    phi0 = 0.0
    r0 = np.array([(R + r_offset)*np.cos(phi0), (R + r_offset)*np.sin(phi0), 0.0])

    # Mostly toroidal (parallel) velocity, plus a small vertical (perpendicular)
    # kick. A packet launched exactly along the field line has zero initial
    # Lorentz force (v parallel to B) and never starts gyrating -- an
    # unrealistic edge case, so we nudge it slightly off-axis in velocity.
    v_par = speed * np.array([-np.sin(phi0), np.cos(phi0), 0.0])
    v_perp = np.array([0.0, 0.0, speed * perp_frac])
    pos, vel = r0.copy(), (v_par + v_perp)

    period = (2*np.pi*R) / speed
    dt_local = period / steps_per_orbit
    n_steps_local = n_orbits * steps_per_orbit

    r_minor_hist = np.zeros(n_steps_local)
    t = 0.0

    def accel_local(r_vec, v_vec):
        return q_over_m * np.cross(v_vec, B_func(r_vec[0], r_vec[1], r_vec[2]))

    for i in range(n_steps_local):
        r_minor_hist[i] = r_minor_at(pos)

        a1 = accel_local(pos, vel); k1_v = a1*dt_local; k1_r = vel*dt_local
        a2 = accel_local(pos+0.5*k1_r, vel+0.5*k1_v); k2_v = a2*dt_local; k2_r = (vel+0.5*k1_v)*dt_local
        a3 = accel_local(pos+0.5*k2_r, vel+0.5*k2_v); k3_v = a3*dt_local; k3_r = (vel+0.5*k2_v)*dt_local
        a4 = accel_local(pos+k3_r, vel+k3_v); k4_v = a4*dt_local; k4_r = (vel+k3_v)*dt_local

        vel = vel + (k1_v + 2*k2_v + 2*k3_v + k4_v)/6.0
        pos = pos + (k1_r + 2*k2_r + 2*k3_r + k4_r)/6.0
        t += dt_local

    return r_minor_hist, period

### Putting numbers on it

Now the lap-test runs twice: once with the field from before (toroidal only) and once with the poloidal twist added, from identical starting conditions -- so any difference in outcome is down to the field shape alone.

In [ ]:
# --- Run: toroidal-only vs toroidal+poloidal, same launch conditions ---
#
# NOTE ON PARAMETERS: the earlier gate-escape cell used q_over_m = 5e5,
# which sits between an electron and a proton and isn't a real species --
# it was tuned to make a nice-looking trajectory. The plasma here is
# helium-sourced, so we use helium's actual charge-to-mass ratio instead.
# Helium can be singly or doubly ionized; doubly-ionized (He2+, alpha-like,
# the fully-stripped state typical of hot plasma) is used below, but both
# were checked -- see the note in the markdown cell after the plot.

q_over_m = 4.8211e7   # He2+ (doubly ionized helium-4), C/kg
speed    = 5.0e4      # m/s

gyro = speed / (q_over_m * np.linalg.norm(B_field_fast(R+0.01, 0.0, 0.0)))
print(f"Gyroradius: {gyro:.4f} m  (channel radius r_plasma = {r_plasma} m)")

r_minor_toroidal_only, period = run_orbit(B_field_fast, q_over_m, speed, n_orbits=25, steps_per_orbit=250)
r_minor_confined, _          = run_orbit(B_confine,    q_over_m, speed, n_orbits=25, steps_per_orbit=250)

print(f"Toroidal-only : start={r_minor_toroidal_only[0]:.4f}  end={r_minor_toroidal_only[-1]:.4f}  max={r_minor_toroidal_only.max():.4f}")
print(f"Toroidal+pol. : start={r_minor_confined[0]:.4f}  end={r_minor_confined[-1]:.4f}  max={r_minor_confined.max():.4f}")

In [ ]:
import matplotlib.pyplot as plt

n_orbits_plot, steps_per_orbit = 25, 250
orbit_axis = np.linspace(0, n_orbits_plot, n_orbits_plot*steps_per_orbit)

plt.figure(figsize=(8,4))
plt.plot(orbit_axis, r_minor_toroidal_only, label="Toroidal field only", color="orange")
plt.plot(orbit_axis, r_minor_confined, label="Toroidal + poloidal (confined)", color="cyan")
plt.axhline(r_plasma, color="grey", linestyle="--", label="Plasma channel edge")
plt.xlabel("Orbit number")
plt.ylabel("Distance from magnetic axis, r_minor (m)")
plt.title("Confinement survival test: does the packet stay inside the channel?")
plt.legend()
plt.tight_layout()
plt.show()

### What this shows (verified numerically before writing this cell)

- **Toroidal field only:** `r_minor` grows without bound -- by orbit 25 the packet is roughly 19 m from the magnetic axis for He2+, i.e. long gone. Confirms the textbook result: a purely toroidal field cannot confine a particle, gate or no gate.
- **Toroidal + poloidal (`I_p = 5e4` A), He2+:** `r_minor` stays bounded between roughly 0.007 m and 0.010 m -- comfortably inside the 0.03 m channel, for the full 25-orbit run.
- **Helium ionization state was checked both ways.** He2+ has a gyroradius of ~0.0159 m here (about half the channel radius); He+ has ~0.0318 m (slightly *larger* than the channel). Despite that difference, both confine cleanly under the same `I_p = 5e4` A poloidal field (He+: end 0.0082 m, max 0.0100 m) -- reassuring, since it means the confinement result isn't fragile to exactly how ionized the helium is.
- **`I_p` still matters a lot, and isn't just "more is better."** `I_p` between ~1e4 and ~1e5 A confines cleanly at these parameters; pushed up to 3e5 A the orbit destabilises again. This is the same territory as a tokamak's *safety factor* `q = (r * B_phi) / (R * B_pol)` -- certain ratios of toroidal to poloidal field drive resonant instabilities (kink modes) rather than better confinement. Worth keeping in mind before assuming "stronger poloidal field = safer" as this gets extended.
- **Next step once this holds:** layer in the betatron flux-ramp (accelerate the confined packet across orbits) *before* adding RF heating or the magnetic mirror cones -- there's no point tuning those on top of a confinement loop that hasn't been shown to hold a packet steady first.

### In plain terms: what this proves, and what's still open

- A ring-shaped magnet on its own **cannot** hold onto a charged particle -- it always leaks out sideways, gate or not. That's not a flaw in this particular design; it's true of any purely toroidal field.
- Adding a second, twisting field component **does** fix this, at least for the packet and timescales tested here -- it stayed put instead of drifting away.
- "Twist strength" has a sweet spot, not a "more is always better" relationship -- too much twist relative to the main field caused the same kind of loss as having no twist at all. Real fusion reactors run into exactly this tradeoff, and spend a lot of engineering effort staying inside the safe range.
- The next building block (speeding the packet up over many laps before release) has to be added **on top of** a version of this that's already confirmed to hold the packet steady -- there's no point tuning the acceleration or the gate timing on a trap that leaks.